> **Solutions.** This is the tutorial notebook with every exercise filled in.
> Try them yourself in `from_urdf_to_semantic_world.ipynb` first.

# From Scene Descriptions to a Semantic Digital Twin

*IJCAI 2026 workshop — hands-on tutorial (150 minutes)*

A scene description tells a robot **where** every link is. It does not tell the robot **what** any of
them is. `<link name="iai_fridge_door"/>` is a name a human chose; to the robot it is a
rigid body on a revolute joint, indistinguishable from a cabinet door, a car door, or a
telescopic mast.

This tutorial builds that understanding from the ground up, in the
[Semantic Digital Twin](https://github.com/cram2/cognitive_robot_abstract_machine), a world
model that carries geometry, kinematics **and** meaning in one structure.

| § | What we do                                                                             |
|---|----------------------------------------------------------------------------------------|
| 1 | What a **World** is: bodies, connections, transformations.                             |
| 2 | What **semantics** adds on top.                                                        |
| 3 | Load an existing world from a URDF, and let the `WorldReasoner` tell you what is in it |
| 4 | Import objects into a world you loaded.                                                |
| 5 | Import a robot, inspect what it is made of, and move one of its joints                 |

Every chapter from §3 on parses its own fresh world, nothing carries over from the chapter
before it, so what you see in RViz always matches the cell you just ran, not some earlier one.

By the end you can take *any* supported scene description, load it, find out what is actually in it, put things into
it, open what needs opening, and bring a robot into the scene, the whole loop this tutorial is
named after.

## 0. Setup

We will look at worlds through **RViz2** instead of an inline notebook widget, the same tool
you would point at a real robot. Open RViz2 now (`rviz2` in a terminal with the workspace
sourced), and add:

- a **TF** display, with the fixed frame set to `map`,
- a **MarkerArray** display, subscribed to `/semworld/viz_marker`.

Leave both running for the rest of the notebook, every world we build below publishes to the
same topic, so RViz always shows whichever world you last looked at.

Run the cell below. It should print a version and a few `OK` lines.

In [1]:
import logging
import threading
from collections import Counter
from pathlib import Path
from importlib.resources import files

import numpy as np
import rclpy

import semantic_digital_twin
from semantic_digital_twin.adapters.package_resolver import CompositePathResolver
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.api import (
    BodySpecification,
    Connection6DoFSpecification,
    RobotSpecification,
    SemanticAnnotationWithRootSpecification,
)
from semantic_digital_twin.exceptions import ExerciseVerificationFailed
from semantic_digital_twin.reasoning.world_reasoner import WorldReasoner
from semantic_digital_twin.robots.pr2 import PR2, PR2Joint
from semantic_digital_twin.semantic_annotations.semantic_annotations import Drawer, Fridge, Milk
from semantic_digital_twin.spatial_types.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.connections import Connection6DoF
from semantic_digital_twin.world_description.geometry import Color, Scale

logging.disable(logging.CRITICAL)  # keep the notebook output readable

print("semantic_digital_twin", semantic_digital_twin.__version__)

RESOURCES_DIR = Path(files("semantic_digital_twin")).parent.parent / "resources"
URDF_DIR = RESOURCES_DIR / "urdf"
MILK_MESH = RESOURCES_DIR / "stl" / "milk.stl"
print("OK  resources dir:", RESOURCES_DIR)

# The kitchen URDFs reference their meshes as package://iai_kitchen/... , which comes from
# the iai_maps ROS package. If this line fails, the ROS workspace was not sourced.
CompositePathResolver().resolve("package://iai_kitchen/meshes/misc/Sink.obj")
print("OK  mesh packages resolve")

# One ROS2 node for the whole notebook. Every world we visualize below publishes onto it.
rclpy.init()
node = rclpy.create_node("semantic_digital_twin_tutorial")
threading.Thread(target=rclpy.spin, args=(node,), daemon=True).start()
print("OK  ROS2 node spinning")


def visualize(world: World, topic_name: str = "/semworld/viz_marker") -> None:
    '''Publish `world` to RViz2. Call once per world; further changes to it (moving a
    body, opening a door, ...) are pushed automatically, no need to call this again.'''
    #TFPublisher(_world=world, node=node)
    VizMarkerPublisher(_world=world, node=node, topic_name=topic_name).with_tf_publisher()



/home/sorin/.virtualenvs/cram2-env/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


semantic_digital_twin 26.07.0
OK  resources dir: /home/sorin/cram2/cognitive_robot_abstract_machine/semantic_digital_twin/resources
OK  mesh packages resolve
OK  ROS2 node spinning


## 1. What is a `World`?

A `World` is a graph:

- **Bodies** are the nodes: each one geometry (for collision and for what you see in
  RViz) plus a name. A body knows nothing about being a "fridge" or a "drawer"; it is just a
  shape somewhere.
- **Connections** are the edges: each one says how a child body's pose relates to its
  parent's. A connection can be rigid (`FixedConnection`: the child never moves relative to
  its parent) or it can carry one or more **degrees of freedom** (a `PrismaticConnection`
  slides along an axis, a `RevoluteConnection` rotates about one, a `Connection6DoF` is free
  to translate and rotate in all six — the kind of connection a loose object on a table has to
  the world, not attached to anything in particular).
- **Transformations** place one frame relative to another. Every `HomogeneousTransformationMatrix`
  you write below is a 4×4 matrix; the library's convention, used everywhere, is to read
  `A_T_B` as *"the pose of B, expressed in frame A."* Two things about them are enough for
  what follows:
  - `HomogeneousTransformationMatrix.from_xyz_rpy(x=.., y=.., z=.., roll=.., pitch=.., yaw=..)`
    builds one from a position and Euler angles — everything defaults to `0`.
  - Transforms compose with `@`. If you know `world_T_a` and `a_T_b`, then `world_T_a @ a_T_b`
    is `world_T_b` — the pose of `b`, expressed in `world` (for more informations visit: https://cram2.github.io/cognitive_robot_abstract_machine/semantic_digital_twin/style_guide.html) .

Nothing here says what anything *means*. First, build a world with your own hands.

In [2]:
world = World.create_with_root_body()
print("root body:", world.root.name.name)  # every fresh world's root is called "map"

table_top = BodySpecification.box(
    name="table_top",
    scale=Scale(1.2, 0.8, 0.05),
    color=Color(0.6, 0.4, 0.2, 1.0),
).spawn(world)  # spawn() with no parent attaches to world.root with a FixedConnection

print("bodies     ", len(world.bodies))
print("connections", Counter(type(c).__name__ for c in world.connections))
print("table_top's connection:", type(table_top.parent_connection).__name__,
      "parent:", table_top.parent_connection.parent.name.name)

visualize(world)

root body: map
bodies      2
connections Counter({'FixedConnection': 1})
table_top's connection: FixedConnection parent: map


`BodySpecification.box(...)` describes a box-shaped body without touching any world —
you can `.spawn()` the same specification into as many worlds as you like. Calling `.spawn`
is what actually creates the `Body`, wires its `FixedConnection` to the parent (`world.root`
by default), and registers everything with `world`.

### A transform, composed

Here is the `@` composition mentioned above, concretely:

In [3]:
world_T_a = HomogeneousTransformationMatrix.from_xyz_rpy(x=1.0)
a_T_b = HomogeneousTransformationMatrix.from_xyz_rpy(y=0.5)
world_T_b = world_T_a @ a_T_b

print("world_T_b position:", world_T_b.to_position().to_np().flatten()[:3])

world_T_b position: [1.  0.5 0. ]


### Exercise 1.1 — place the milk

You are about to build a two-cube "kitchen": two boxes standing in for counters, with a real
milk carton mesh (`MILK_MESH`, defined in §0 — the same asset §4 uses again later) sitting on
top of one of them. Before assembling it, work out where the milk goes.

Both cubes are `cube_side = 0.4` m on a side, and boxes are centered on their own frame, so a
cube spawned with no rotation has its top face `cube_side / 2` above its own origin. The milk
mesh is centered on its own origin. The distance from that origin down to
the mesh's actual base is given as `MILK_BASE_OFFSET = 0.0877` m.

Build a `HomogeneousTransformationMatrix` with `from_xyz_rpy` that has no rotation, no `x` or
`y` offset, and the `z` offset that puts the milk's *base* flush on top of a
`cube_side`-tall cube. Assign it to `milk_offset`.

In [4]:
cube_side = 0.4
MILK_BASE_OFFSET = 0.0877

milk_offset = HomogeneousTransformationMatrix.from_xyz_rpy(z=cube_side / 2 + MILK_BASE_OFFSET)

In [5]:
# Run this to check your answer.
if milk_offset is ... or not isinstance(milk_offset, HomogeneousTransformationMatrix):
    raise ExerciseVerificationFailed(
        "milk_offset should be a HomogeneousTransformationMatrix."
    )

position = milk_offset.to_position().to_np().flatten()[:3]
expected = np.array([0.0, 0.0, cube_side / 2 + MILK_BASE_OFFSET])
if not np.allclose(position, expected, atol=1e-6):
    raise ExerciseVerificationFailed(f"Expected position {expected}, got {position}.")

print("Correct.")

Correct.


### Exercise 1.2 — build the mini kitchen

Now assemble it. Both cubes get a `y=2.0` offset so they land well clear of `table_top`,
which is still parked at the world origin — a different `World` object, but the same RViz
view, so anything spawned near the origin here would clip straight through it.

1. Create a fresh `kitchen_world = World.create_with_root_body()`.
2. Spawn `cube_1 = BodySpecification.box(name="cube_1", scale=Scale(cube_side, cube_side, cube_side)).spawn(kitchen_world, parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(y=2.0))`.
3. Spawn `cube_2` the same way, but `parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=1.0, y=2.0)`
   so it stands a meter away from `cube_1` (and at the same `y`), not on top of it.
4. Spawn a `Milk` semantic annotation on top of `cube_1`, using `milk_offset` from Exercise 1.1
   and `SemanticAnnotationWithRootSpecification`, now with a real mesh instead of a box:

   ```python
   milk = SemanticAnnotationWithRootSpecification(
       name="milk",
       semantic_annotation_type=Milk,
       root_specification=BodySpecification.mesh(
           name="milk_body",
           filename=str(MILK_MESH),
           connection_specification=Connection6DoFSpecification(),
       ),
   ).spawn(kitchen_world, parent=cube_1, parent_T_self=milk_offset)
   ```

   `Connection6DoFSpecification()` matters here: a carton of milk is not bolted to the
   counter, so it gets a connection with all six degrees of freedom instead of the default
   `FixedConnection` — that is what makes Exercise 1.3 (moving it) possible at all.
5. Visualize `kitchen_world`.

In [7]:
kitchen_world = World.create_with_root_body()

cube_1 = BodySpecification.box(
    name="cube_1", scale=Scale(cube_side, cube_side, cube_side),
).spawn(kitchen_world, parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(y=2.0))

cube_2 = BodySpecification.box(
    name="cube_2", scale=Scale(cube_side, cube_side, cube_side),
).spawn(kitchen_world, parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=1.0, y=2.0))

milk = SemanticAnnotationWithRootSpecification(
    name="milk",
    semantic_annotation_type=Milk,
    root_specification=BodySpecification.mesh(
        name="milk_body",
        filename=str(MILK_MESH),
        connection_specification=Connection6DoFSpecification(),
    ),
).spawn(kitchen_world, parent=cube_1, parent_T_self=milk_offset)

visualize(kitchen_world)

In [8]:
# Run this to check your answer.
if kitchen_world is ... or not isinstance(kitchen_world, World):
    raise ExerciseVerificationFailed("kitchen_world should be a World.")
for name, body in [("cube_1", cube_1), ("cube_2", cube_2)]:
    if body is ... or body not in kitchen_world.bodies:
        raise ExerciseVerificationFailed(f"{name} should be a Body spawned into kitchen_world.")
if milk is ... or not isinstance(milk, Milk):
    raise ExerciseVerificationFailed("milk should be the Milk annotation you spawned.")
if milk.root.parent_connection.parent is not cube_1:
    raise ExerciseVerificationFailed("milk should be attached to cube_1.")
if not isinstance(milk.root.parent_connection, Connection6DoF):
    raise ExerciseVerificationFailed(
        "milk's root_specification should set connection_specification=Connection6DoFSpecification()."
    )

world_position = milk.root.global_pose.to_position().to_np().flatten()[:3]
expected = np.array([0.0, 2.0, cube_side / 2 + MILK_BASE_OFFSET])
if not np.allclose(world_position, expected, atol=1e-6):
    raise ExerciseVerificationFailed(
        f"milk should sit on top of cube_1, at {expected}; it is at {world_position}."
    )

print("Correct. Check RViz: two cubes, milk on the left one.")

Correct. Check RViz: two cubes, milk on the left one.


### Exercise 1.3 — move the milk to the other cube

Bodies are stuck to the parent they were spawned with, if it is not re-parented. `World.move_branch` re-parents a
body while keeping its *world* pose, the milk will not visibly jump yet, it will just now be
kinematically a child of `cube_2` instead of `cube_1`. Then, because the milk's connection is
a `Connection6DoF`, you can write a new pose straight into its `.origin`, the same way you
would write to a body's transform anywhere else in this library:

```python
with kitchen_world.modify_world():
    kitchen_world.move_branch(branch_root=milk.root, new_parent=cube_2)
milk.root.parent_connection.origin = HomogeneousTransformationMatrix.from_xyz_rpy(
    z=cube_side / 2 + MILK_BASE_OFFSET, reference_frame=cube_2,
)
```

(`move_branch` restructures the kinematic tree — adding a connection, in effect — so like every
structural change in this library it has to happen inside a `world.modify_world()` block.
Setting `.origin` afterwards is a state change, not a structural one, so it does not need one.)

Do that, then visualize `kitchen_world` again and check RViz: the milk should now be sitting
on `cube_2`.

In [9]:
with kitchen_world.modify_world():
    kitchen_world.move_branch(branch_root=milk.root, new_parent=cube_2)

milk.root.parent_connection.origin = HomogeneousTransformationMatrix.from_xyz_rpy(
    z=cube_side / 2 + MILK_BASE_OFFSET, reference_frame=cube_2,
)

visualize(kitchen_world)

In [10]:
# Run this to check your answer.
if milk.root.parent_connection.parent is not cube_2:
    raise ExerciseVerificationFailed(
        "milk should now be attached to cube_2 — use kitchen_world.move_branch(...)."
    )

world_position = milk.root.global_pose.to_position().to_np().flatten()[:3]
expected = np.array([1.0, 2.0, cube_side / 2 + MILK_BASE_OFFSET])
if not np.allclose(world_position, expected, atol=1e-6):
    raise ExerciseVerificationFailed(
        f"milk should sit on top of cube_2, at {expected}; it is at {world_position}. "
        "Did you set milk.root.parent_connection.origin?"
    )

print("Correct. Check RViz: the milk should have moved onto the right-hand cube.")

Correct. Check RViz: the milk should have moved onto the right-hand cube.


## 2. Semantic Annotations: What semantics adds

New chapter, new world. Build a small, fresh one with a shelf and a carton of milk on it, then ask it what it *means*:

In [13]:
semantics_world = World.create_with_root_body()

shelf = BodySpecification.box(
    name="shelf", scale=Scale(0.6, 0.3, 0.05), color=Color(0.6, 0.4, 0.2, 1.0),
).spawn(semantics_world)

milk = Milk.get_annotation_specification(
    name="milk",
    root_specification=BodySpecification.mesh(
        name="milk_body",
        filename=str(MILK_MESH),
        connection_specification=Connection6DoFSpecification(),
    ),
).spawn(semantics_world, parent=shelf, parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(
    z=0.025 + MILK_BASE_OFFSET,
))

visualize(semantics_world)
print(semantics_world.semantic_annotations)

[Milk(class_label=None, name=PrefixedName('None/milk'), id=UUID('ced04cc0-668c-4ada-b1dd-274ecb7cedb9'), root=Body(name=PrefixedName('None/milk'), id=UUID('f29099cd-c46c-4853-a8e5-ff44eeca336b'), index=2))]


One entry: `Milk`. Not two, even though there are two bodies in this tiny world. That is the
whole point of a **semantic annotation**: it says that a particular body or a particular
combination of bodies *is* something, in a way a program can act on. `shelf` is a shape with
a name; `milk` is a `Milk`, a Python type you can check with `isinstance`, search for, and
reason about.

You could, of course, have gone looking for "the milk" by name instead:

```python
[b.name.name for b in semantics_world.bodies if "milk" in b.name.name.lower()]
```

That works here, because you happened to name the body `milk_body`. It stops working the
moment someone hands you a URDF where link `piece_047` is, in fact, a carton of milk, which
is exactly the situation a real perception pipeline is in, and exactly why the semantic layer
exists: it is a property of the *body*, not of the string a human chose to type.

### Querying by type

The straightforward way to ask "what `Milk` is in this world" is
`get_semantic_annotations_by_type`, which you have already used once, implicitly, when the
exercise checkers in §1 looked at `milk`:

In [15]:
milks = semantics_world.get_semantic_annotations_by_type(Milk)
print(len(milks), "Milk annotation(s):", [milk.name for milk in milks])

1 Milk annotation(s): [PrefixedName('None/milk')]


That is the only query mechanism this tutorial uses, and it is enough for everything below.
For anything past "give me all of type X", filtering on attributes, following part-whole
relationships, questions that span several objects, the library ships a small, typed query
builder called the **Entity Query Language** (EQL), from the `krrood` package. It is out of
scope here; the further-reading table at the end of this notebook links to its docs. Also note that there will be a tutorial on EQL tomorrow.

Two cubes and one carton of milk is a toy example, the payoff is that the query above does
not change at all when the world gets bigger. §3 loads a URDF with 80-odd bodies in it, and
asks it the exact same kind of question.

## 3. Importing an existing world, and asking it questions

Nobody hand-builds a kitchen body by body. `URDFParser` reads an existing URDF file into a
`World`, same kind of object as `kitchen_world` above, just assembled from a file instead of
from `BodySpecification` calls.

> **Switch RViz's Fixed Frame.** This world's root is not called `map` — a URDF brings its own
> root link name along with it. In RViz2, **Displays → Global Options → Fixed Frame**, change
> it from `map` to `iai_oven_area/world`. Everything for the rest of this notebook lives in
> this one world, so this is the last frame switch you need to make.

In [19]:
KITCHEN_SMALL = URDF_DIR / "kitchen-small.urdf"

kitchen = URDFParser.from_file(str(KITCHEN_SMALL)).parse()

print("bodies     ", len(kitchen.bodies))
print("connections", Counter(type(c).__name__ for c in kitchen.connections))
print("semantic annotations:", kitchen.semantic_annotations)
print("root body:", kitchen.root.name)

visualize(kitchen)

Scalar element defined multiple times: limit


bodies      77
connections Counter({'FixedConnection': 52, 'PrismaticConnection': 15, 'RevoluteConnection': 9})
semantic annotations: []
root body: iai_oven_area/world


Empty annotations, same as `kitchen_world` was before you built anything into it  a world
parsed straight from a file is purely kinematic. Nothing in it is a fridge or a drawer; those
are things *we* read into the link names, the same trap §2 pointed out.

### The `WorldReasoner`

`WorldReasoner` applies a body of rules to a raw world and infers semantic annotations for
you, the same way you built the `Milk` annotation by hand in §1 except now for every fridge,
door, drawer and handle a rule can recognize, in one call.

In [20]:
reasoner = WorldReasoner(kitchen)
inferred = reasoner.reason()["semantic_annotations"]

print(f"{len(inferred)} annotations inferred")
print(Counter(type(a).__name__ for a in inferred))

40 annotations inferred
Counter({'Handle': 18, 'Drawer': 14, 'Wardrobe': 4, 'Door': 3, 'Fridge': 1})


The rules are structural, not name-based — "a body that slides, with a handle rigidly
attached to it, is a drawer" — so they still work on a URDF where every link is called
`link_042`. If you want to see a rule explain itself, the reasoner can show its work:

```python
from krrood.entity_query_language.explanation.explanation import explain_inference
from krrood.entity_query_language.verbalization.pipeline import verbalize_expression

drawers = kitchen.get_semantic_annotations_by_type(Drawer)
explanation = explain_inference(drawers[0])
print(verbalize_expression(explanation.query_root))
```

### Exercise 3.1 — find the fridge

Use `get_semantic_annotations_by_type` to get the `Fridge` annotation out of `kitchen`, and
assign it to `fridge`. Then print how many doors it has and the name of its handle, e.g.
`fridge.doors[0].handle.root.name.name`.

In [21]:
fridge = kitchen.get_semantic_annotations_by_type(Fridge)[0]

print(len(fridge.doors), fridge.doors[0].handle.root)

1 Body(name=PrefixedName('iai_oven_area/iai_fridge_door_handle'), id=UUID('fb088a30-ed44-4de9-8b3b-9b9c21af166f'), index=75)


In [22]:
# Run this to check your answer.
if fridge is ... or not isinstance(fridge, Fridge):
    raise ExerciseVerificationFailed("fridge should be the Fridge annotation the reasoner found.")
if not fridge.doors:
    raise ExerciseVerificationFailed("fridge.doors should not be empty.")
if fridge.doors[0].handle is None:
    raise ExerciseVerificationFailed("fridge.doors[0] should have a handle.")

print("Correct:", fridge.root.name.name, "has", len(fridge.doors), "door(s).")

Correct: iai_fridge_main has 1 door(s).


## 4. Importing objects, and opening what's in the way

`kitchen` is still the world from §3, `fridge` still the annotation you found in Exercise 3.1.
The last piece is putting things into it — the same
`BodySpecification`/`SemanticAnnotationWithRootSpecification` calls from §1, now aimed at a
body the reasoner found instead of one you built by hand.

Put a milk carton — same `MILK_MESH` from §1 — near the fridge. Its placement here is a guess,
on purpose:

In [ ]:
milk = SemanticAnnotationWithRootSpecification(
    name="milk",
    semantic_annotation_type=Milk,
    root_specification=BodySpecification.mesh(
        name="milk_body",
        filename=str(MILK_MESH),
        connection_specification=Connection6DoFSpecification(),
    ),
).spawn(
    kitchen,
    parent=fridge.root,
    # A guess — we don't know the fridge's exact extent, so this just clears its top.
    parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(z=1.8),
)

visualize(kitchen)

Check RViz: the milk is floating a meter above the fridge. `parent=fridge.root` came from a
query, not a name you had to already know — that is what "importing an object" means — but
nothing is holding it up at that height, because `z=1.8` was a guess.

### Putting it somewhere real

Fix that by moving the *same* milk onto an actual surface, the way you moved it between cubes
in Exercise 1.3. This fridge model has no drawer of its own, but the reasoner found plenty
elsewhere in `kitchen`:

In [ ]:
drawer = kitchen.get_semantic_annotations_by_type(Drawer)[0]

with kitchen.modify_world():
    kitchen.move_branch(branch_root=milk.root, new_parent=drawer.root)

milk.root.parent_connection.origin = HomogeneousTransformationMatrix.from_xyz_rpy(
    z=0.1, reference_frame=drawer.root,
)

visualize(kitchen)

Check RViz: the same milk carton — one `Milk` annotation the whole time, never spawned
twice — should now be sitting inside the drawer instead of floating.

### Opening the drawer

§1 already showed the mechanism: writing to a connection's DOF-backed `.position` (or, for a
`Connection6DoF`, `.origin`) changes where its child sits, and every child further down the
tree follows. A `PrismaticConnection` (a drawer's slider) is DOF-backed the same way a
`Connection6DoF` is — so opening the drawer the milk is now sitting in is the same operation
again, on `drawer.root.parent_connection` this time:

In [ ]:
drawer_slider = drawer.root.parent_connection

print(type(drawer_slider).__name__, "range:",
      drawer_slider.dof.limits.lower.position, "->", drawer_slider.dof.limits.upper.position)

drawer_slider.position = 0.9 * drawer_slider.dof.limits.upper.position

visualize(kitchen)

Check RViz: the drawer should be open, milk sliding out with it — unprompted, because it is a
kinematic child of `drawer.root`. Nothing else about the world changed for this; opening a
drawer is a *state* change, and the `TFPublisher` registered back in §0 pushes state changes on
its own.

### Exercise 4.1 — milk in the fridge

Same two moves — spawn an object where it doesn't belong yet, then open the mechanism standing
between it and the world — but on the fridge itself instead of the drawer:

1. Spawn a second `Milk` — reuse `BodySpecification.mesh`/`MILK_MESH` and
   `Connection6DoFSpecification()` — as a child of `fridge.root`, roughly where the inside of
   the fridge would be. Assign it to `milk_in_fridge`.
2. `fridge.doors[0]` is a `Door`, and `Door.root.parent_connection` is its hinge — a
   `RevoluteConnection`, not a `PrismaticConnection`, but DOF-backed the exact same way. Assign
   it to `fridge_hinge` and set its `.position` to something past the midpoint of its range,
   closer to `dof.limits.upper.position` than to `0`.
3. Visualize `kitchen` and check RViz.

In [ ]:
milk_in_fridge = SemanticAnnotationWithRootSpecification(
    name="milk_in_fridge",
    semantic_annotation_type=Milk,
    root_specification=BodySpecification.mesh(
        name="milk_in_fridge_body",
        filename=str(MILK_MESH),
        connection_specification=Connection6DoFSpecification(),
    ),
).spawn(
    kitchen, parent=fridge.root,
    # In fridge.root's own local frame, not world: (0, 0, 0) is roughly the fridge's own
    # center, and its housing extends about half a meter each way, so this sits comfortably
    # inside the cabinet rather than on top of it.
    parent_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.1, y=-0.1, z=0.05),
)

fridge_hinge = fridge.doors[0].root.parent_connection
fridge_hinge.position = 0.9 * fridge_hinge.dof.limits.upper.position

visualize(kitchen)

In [ ]:
# Run this to check your answer.
if milk_in_fridge is ... :
    raise ExerciseVerificationFailed("milk_in_fridge should be the annotation you spawned.")
if milk_in_fridge.root.parent_connection.parent is not fridge.root:
    raise ExerciseVerificationFailed("milk_in_fridge should be attached to fridge.root.")
if not isinstance(milk_in_fridge.root.parent_connection, Connection6DoF):
    raise ExerciseVerificationFailed(
        "Give milk_in_fridge a Connection6DoFSpecification() so it is a movable object, not bolted on."
    )

expected_connection = fridge.doors[0].root.parent_connection
if fridge_hinge is not expected_connection:
    raise ExerciseVerificationFailed(
        "fridge_hinge should be fridge.doors[0].root.parent_connection."
    )
lower = fridge_hinge.dof.limits.lower.position
upper = fridge_hinge.dof.limits.upper.position
if not (lower <= fridge_hinge.position <= upper):
    raise ExerciseVerificationFailed(f"fridge_hinge.position must be within {lower} to {upper}.")
if fridge_hinge.position < lower + 0.5 * (upper - lower):
    raise ExerciseVerificationFailed(
        "Open the door further — set fridge_hinge.position closer to its upper limit."
    )

print("Correct. Check RViz: the fridge door should be open, with milk inside it.")
visualize(kitchen)

## 5. Robots

New chapter, fresh world — same rule as §4: `kitchen` from here on has none of the previous
chapter's door-opening or milk-carton state in it, just a clean parse of the same file.

A robot is merged into a world the same way a URDF-parsed scene is — `RobotSpecification`
parses the robot's own description and attaches it to `world.root` via a mobile base, at
whatever pose you give its `odom` frame. Bring a PR2 into a fresh `kitchen`, standing in front
of its fridge:

In [ ]:
kitchen = URDFParser.from_file(str(KITCHEN_SMALL)).parse()
WorldReasoner(kitchen).reason()

pr2 = RobotSpecification(
    semantic_annotation_type=PR2,
    world_T_odom=HomogeneousTransformationMatrix.from_xyz_rpy(x=0.3, y=-2.3),
).spawn(kitchen)

print("pr2 is a semantic annotation of type:", type(pr2).__name__)
print("bodies in kitchen now:", len(kitchen.bodies))

visualize(kitchen)

### Inspecting a robot

`pr2` decomposes into typed parts the same way `fridge` decomposed into doors and handles:
`pr2.torso`, `pr2.left_arm`, `pr2.right_arm`, each a robot part with its own kinematic chain.
Every part exposes `.active_connections` — the connections with a hardware-controlled degree
of freedom, i.e. the joints an actual robot could move:

In [ ]:
for connection in pr2.left_arm.active_connections:
    print(f"{connection.name.name:24s} {type(connection).__name__:18s} "
          f"{connection.dof.limits.lower.position} -> {connection.dof.limits.upper.position}")

### Moving a joint

Same mechanism as §4 — a connection with a DOF, and you write to `.position`. You can look a
connection up by name directly with `world.get_connection_by_name`, which is what
`PR2Joint` — an enum of the PR2's commandable joint names — is for:

In [ ]:
torso_lift = kitchen.get_connection_by_name(PR2Joint.TORSO_LIFT)
print("torso range:", torso_lift.dof.limits.lower.position, "->", torso_lift.dof.limits.upper.position)

torso_lift.position = 0.9 * torso_lift.dof.limits.upper.position

visualize(kitchen)

Check RViz: the PR2's torso should be raised. Everything below it in the kinematic tree — the
arms, the head — moved with it, the same way the milk moved with the drawer in §4.

### Cleanup

When you are done, shut down the ROS2 node cleanly.

In [ ]:
node.destroy_node()
rclpy.shutdown()

## Where to go next

What we did: built a world from three primitive shapes to see what bodies, connections and
transformations actually are; layered typed semantic annotations on top and queried them by
type; loaded a real URDF and let a `WorldReasoner` find its fridges and drawers without being
told any link names; used what it found as the attachment point for objects we imported
ourselves, and opened what stood between them and the world by writing to a connection's
`.position`; and brought a robot into the scene, inspected what it is made of, and moved its
joints the same way. Given any URDF now, you can load it, find out what is in it, put things
into it, open what needs opening, and bring a robot in to work with it.

What we deliberately left out — planning a collision-free reach, grasping an object with a
gripper, coordinating a whole pick-and-place action — builds on exactly this representation,
one layer up, and is where the next notebook in this series picks up.

The library ships a full Jupyter Book under
`cognitive_robot_abstract_machine/semantic_digital_twin/doc/` (17 worked examples, concept
chapters, and self-assessment quizzes):

| Topic | Guide |
|---|---|
| Transforms and the `A_T_B` convention | `examples/using_transformations.md` |
| Loading worlds from files (URDF, MJCF, STL) | `examples/loading_worlds.md` |
| Declarative world building with specifications | `examples/building_worlds_with_specifications.md` |
| Semantic annotations, in depth | `examples/semantic_annotations.md` |
| Visualizing worlds (RViz2 and simulation) | `examples/visualizing_worlds.md` |
| The Entity Query Language, in depth | [krrood/eql/intro](https://cram2.github.io/cognitive_robot_abstract_machine/krrood/eql/intro.html) |
| Regions and supporting surfaces | `examples/regions.md` |
| Saving annotated worlds to SQL | `examples/persistence_of_annotated_worlds.md` |
| Physics simulation (MuJoCo) | `examples/physics_simulators.md` |
| Adding a new robot | `examples/adding_robots.md` |
| Free-space decomposition and path planning | `examples/graph_of_convex_sets.md` |
| Loading RoboCasa / ProcTHOR / PartNet scenes | `doc/datasets.md` |

To convert any of them into a runnable notebook:

```bash
jupytext --to notebook cognitive_robot_abstract_machine/semantic_digital_twin/examples/regions.md
```

The predecessor to this tutorial, on writing the URDF itself, is
[EASE Fall School 2024 — Creating an Environment URDF](https://github.com/IntEL4CoRo/ease_fall_school_2024).